In [ ]:
!pip install -q --no-deps facenet-pytorch

In [ ]:
import os
import glob
from pathlib import Path
from PIL import Image
import torchvision.transforms as T

# ==============================================================================
# Configuration
# ==============================================================================
# Detect if running in Kaggle or locally
if os.path.exists("/kaggle"):
    # Target the raw dataset, not the MTCNN one, so MTCNN can process the augmented images later
    DATASET_DIR = "/kaggle/input/datasets/mrsohel/autism-dataset/dataset/train"
    OUT_DIR = "/kaggle/working/dataset_augmented/train"
else:
    _repo = Path.cwd().parent
    DATASET_DIR = str(_repo / "dataset" / "train")
    OUT_DIR = str(_repo / "dataset_augmented" / "train")

# We want all classes to have at least this many images
TARGET_COUNT = 600

# Strong augmentations to create diverse synthetic copies
augment_pipeline = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=20),
    T.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.8, 1.2), shear=10),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    T.RandomPerspective(distortion_scale=0.2, p=0.5),
])

def augment_class(class_name, target_count):
    src_dir = Path(DATASET_DIR) / class_name
    dst_dir = Path(OUT_DIR) / class_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    
    if not src_dir.exists():
        print(f"Skipping {class_name}, source dir not found.")
        return

    images = list(src_dir.glob("*.jpg")) + list(src_dir.glob("*.png")) + list(src_dir.glob("*.jpeg"))
    current_count = len(images)
    
    if current_count == 0:
        print(f"No images found for {class_name}.")
        return

    # First, copy all original images
    for img_path in images:
        img = Image.open(img_path).convert("RGB")
        img.save(dst_dir / img_path.name)
        
    print(f"[{class_name.upper()}] Copied {current_count} original images.")
    
    if current_count >= target_count:
        print(f"[{class_name.upper()}] Already has {current_count} images (target: {target_count}). No augmentation needed.")
        return
        
    # Generate new images until we hit the target count
    needed = target_count - current_count
    print(f"[{class_name.upper()}] Generating {needed} synthetic augmented images...")
    
    generated = 0
    while generated < needed:
        # Loop through existing images and create variations
        for img_path in images:
            if generated >= needed:
                break
                
            img = Image.open(img_path).convert("RGB")
            # Apply random transformation
            aug_img = augment_pipeline(img)
            
            # Save new image
            new_name = f"aug_{generated}_{img_path.name}"
            aug_img.save(dst_dir / new_name)
            generated += 1

    print(f"[{class_name.upper()}] Finished! Total images now: {len(list(dst_dir.glob('*')))}")

def copy_unmodified_splits():
    """Copy the valid and test splits unmodified so the full dataset structure is intact."""
    import shutil
    for split in ["valid", "test"]:
        src = Path(DATASET_DIR).parent / split
        dst = Path(OUT_DIR).parent / split
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"[*] Copied {split} split unmodified.")

if __name__ == "__main__":
    print("="*60)
    print(" OFFLINE DATA AUGMENTATION (CLASS BALANCING)")
    print("="*60)
    
    # We explicitly want to boost the minority classes up to ~600 (like Joy)
    classes_to_process = ["anger", "fear", "joy", "natural", "sadness", "surprise"]
    
    for c in classes_to_process:
        augment_class(c, TARGET_COUNT)
        
    copy_unmodified_splits()
    
    print("\n[*] Offline augmentation complete!")
    print(f"[*] Your balanced raw dataset is now located at: {Path(OUT_DIR).parent}")
    print("[*] NEXT STEP: Update preprocess_faces.py to point RAW_DATASET to this new folder, then re-run it.")


In [ ]:
import os
import sys
import argparse
import warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
from PIL import Image

warnings.filterwarnings("ignore")


# ==============================================================================
# Paths  (auto-detects Kaggle vs local)
# ==============================================================================
if os.path.exists("/kaggle"):
    RAW_DATASET = "/kaggle/working/dataset_augmented"
    OUT_DATASET = "/kaggle/working/dataset_mtcnn"
else:
    _repo       = Path(__file__).resolve().parent.parent   # repo root
    RAW_DATASET = str(_repo / "dataset")
    OUT_DATASET = str(_repo / "dataset_mtcnn")

SPLITS   = ["train", "valid", "test"]
CLASSES  = ["anger", "fear", "joy", "natural", "sadness", "surprise"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

# ==============================================================================
# Config
# ==============================================================================
FACE_PADDING  = 0.20   # fractional padding added around detected face box
FALLBACK_FRAC = 0.85   # centre-crop fraction when no face found
CLAHE_CLIP    = 2.0    # CLAHE clipLimit  (higher = stronger contrast boost)
CLAHE_TILE    = 8      # CLAHE tileGridSize in pixels
OUTPUT_SIZE   = 224    # final image size saved to disk
MIN_FACE_PX   = 30     # reject detections smaller than this (false positives)


# ==============================================================================
# MTCNN — lazy import
# ==============================================================================
def load_mtcnn(device="cpu"):
    try:
        from facenet_pytorch import MTCNN
        mtcnn = MTCNN(
            keep_all=True,                    # detect all faces; we pick the best
            min_face_size=MIN_FACE_PX,
            thresholds=[0.6, 0.7, 0.7],      # P-Net / R-Net / O-Net thresholds
            post_process=False,
            device=device,
        )
        print(f"[OK] MTCNN loaded on {device}")
        return mtcnn
    except ImportError:
        print("[!] facenet-pytorch not installed.")
        print("    Run:  pip install facenet-pytorch")
        print("    Falling back to centre-crop only (no face detection).")
        return None


# ==============================================================================
# Step 1 — Face detection & crop
# ==============================================================================
def detect_and_crop(img_pil, mtcnn):
    """
    Returns (cropped_PIL_image, status_string).
    status is one of: 'detected' | 'fallback_no_face' | 'fallback_small'
    """
    w, h = img_pil.size

    if mtcnn is not None:
        try:
            boxes, probs, landmarks = mtcnn.detect(img_pil, landmarks=True)
        except Exception:
            boxes, probs, landmarks = None, None, None

        if boxes is not None and len(boxes) > 0 and landmarks is not None:
            # Choose face with highest detection confidence
            best = int(np.argmax(probs))
            
            # --- ALIGNMENT STEP ---
            pts = landmarks[best]
            if pts is not None:
                left_eye, right_eye = pts[0], pts[1]
                dy = right_eye[1] - left_eye[1]
                dx = right_eye[0] - left_eye[0]
                angle = np.degrees(np.arctan2(dy, dx))
                
                # Rotate image to make eyes horizontal
                if abs(angle) > 1.5:  # Only rotate if tilt is noticeable
                    eye_center = tuple(((left_eye + right_eye) / 2.0).astype(float))
                    # PIL rotation is counter-clockwise. A positive dy means right eye is lower than left eye.
                    # np.arctan2(dy, dx) is positive. To fix it, we must rotate clockwise (negative angle).
                    # Wait, PIL's rotate takes positive angle for counter-clockwise. 
                    # If right eye is lower (dy>0), angle>0. We need to rotate clockwise, so we pass angle.
                    img_pil = img_pil.rotate(angle, center=eye_center, resample=Image.BICUBIC)
                    
                    # Re-detect on the aligned image to get straight bounding boxes
                    try:
                        boxes2, probs2 = mtcnn.detect(img_pil)
                        if boxes2 is not None and len(boxes2) > 0:
                            best = int(np.argmax(probs2))
                            boxes = boxes2
                    except Exception:
                        pass # Fallback to unaligned boxes (they will be slightly off but acceptable)

            x1, y1, x2, y2 = boxes[best]
            bw, bh = x2 - x1, y2 - y1

            # Reject implausibly small detections
            if bw < MIN_FACE_PX or bh < MIN_FACE_PX:
                return _centre_crop(img_pil), "fallback_small"

            # Add padding around the face
            x1 = max(0,  x1 - bw * FACE_PADDING)
            y1 = max(0,  y1 - bh * FACE_PADDING)
            x2 = min(w,  x2 + bw * FACE_PADDING)
            y2 = min(h,  y2 + bh * FACE_PADDING)

            return img_pil.crop((x1, y1, x2, y2)), "detected"

    return _centre_crop(img_pil), "fallback_no_face"


def _centre_crop(img_pil):
    w, h  = img_pil.size
    cw    = int(w * FALLBACK_FRAC)
    ch    = int(h * FALLBACK_FRAC)
    x0    = (w - cw) // 2
    y0    = (h - ch) // 2
    return img_pil.crop((x0, y0, x0 + cw, y0 + ch))


# ==============================================================================
# Step 2 — CLAHE contrast normalisation (L channel of LAB colour space)
# ==============================================================================
def apply_clahe(img_pil):
    img_np  = np.array(img_pil.convert("RGB"))
    img_lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    clahe   = cv2.createCLAHE(clipLimit=CLAHE_CLIP,
                               tileGridSize=(CLAHE_TILE, CLAHE_TILE))
    img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])   # only L channel
    result  = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
    return Image.fromarray(result)


# ==============================================================================
# Main loop
# ==============================================================================
def process_dataset(raw_root, out_root, mtcnn, use_clahe=True):
    raw_root = Path(raw_root)
    out_root = Path(out_root)

    print(f"\n{'='*64}")
    print(f"  Source  : {raw_root}")
    print(f"  Output  : {out_root}")
    print(f"  CLAHE   : {'ON' if use_clahe else 'OFF'}")
    print(f"  Padding : {int(FACE_PADDING*100)}%  |  Fallback crop: {int(FALLBACK_FRAC*100)}%")
    print(f"{'='*64}\n")

    total_stats = defaultdict(int)

    for split in SPLITS:
        split_stats = defaultdict(int)

        for cls_name in CLASSES:
            src_dir = raw_root / split / cls_name
            dst_dir = out_root / split / cls_name
            dst_dir.mkdir(parents=True, exist_ok=True)

            if not src_dir.exists():
                continue

            img_paths = [p for p in src_dir.iterdir()
                         if p.suffix.lower() in IMG_EXTS]

            for img_path in img_paths:
                dst_path = dst_dir / img_path.name

                # Re-run safe: skip already-processed images
                if dst_path.exists():
                    split_stats["skipped"] += 1
                    continue

                try:
                    img = Image.open(img_path).convert("RGB")

                    # 1. Face detection + crop
                    cropped, status = detect_and_crop(img, mtcnn)
                    split_stats[status] += 1

                    # 2. Optional CLAHE
                    if use_clahe:
                        cropped = apply_clahe(cropped)

                    # 3. Resize and save
                    cropped = cropped.resize((OUTPUT_SIZE, OUTPUT_SIZE),
                                            Image.LANCZOS)
                    cropped.save(dst_path, quality=95)

                except Exception as e:
                    split_stats["error"] += 1
                    # Hard fallback: save resized original
                    try:
                        Image.open(img_path).convert("RGB") \
                             .resize((OUTPUT_SIZE, OUTPUT_SIZE), Image.LANCZOS) \
                             .save(dst_path)
                    except Exception:
                        pass

        # Per-split summary line
        processed = (split_stats["detected"]
                     + split_stats["fallback_no_face"]
                     + split_stats["fallback_small"])
        detected  = split_stats["detected"]
        fallback  = split_stats["fallback_no_face"] + split_stats["fallback_small"]
        det_pct   = detected / max(1, processed) * 100

        print(f"  [{split:5s}]  Processed={processed:4d} | "
              f"Face detected={detected:4d} ({det_pct:5.1f}%) | "
              f"Fallback={fallback:3d} | "
              f"Errors={split_stats['error']:2d} | "
              f"Skipped={split_stats['skipped']:4d}")

        for k, v in split_stats.items():
            total_stats[k] += v

    # Overall summary
    total_processed = (total_stats["detected"]
                       + total_stats["fallback_no_face"]
                       + total_stats["fallback_small"])
    overall_pct = total_stats["detected"] / max(1, total_processed) * 100
    print(f"\n  [TOTAL]  Processed={total_processed} | "
          f"Face detected={total_stats['detected']} ({overall_pct:.1f}%) | "
          f"Fallback={total_stats['fallback_no_face'] + total_stats['fallback_small']} | "
          f"Errors={total_stats['error']}")

    return str(out_root)


# ==============================================================================
# Per-class report (train split only)
# ==============================================================================
def per_class_report(raw_root, out_root):
    raw_root, out_root = Path(raw_root), Path(out_root)
    print(f"\n{'─'*52}")
    print(f"  Per-class images saved (train split)")
    print(f"{'─'*52}")
    for cls_name in CLASSES:
        src = raw_root / "train" / cls_name
        dst = out_root / "train" / cls_name
        n_src = len([p for p in src.iterdir()
                     if p.suffix.lower() in IMG_EXTS]) if src.exists() else 0
        n_dst = len([p for p in dst.iterdir()
                     if p.suffix.lower() in IMG_EXTS]) if dst.exists() else 0
        filled = int(n_dst / max(1, n_src) * 30)
        bar    = "█" * filled + "░" * (30 - filled)
        pct    = n_dst / max(1, n_src) * 100
        print(f"  {cls_name:<10} [{bar}]  {n_dst:3d}/{n_src:3d}  ({pct:.0f}%)")
    print(f"{'─'*52}")


# ==============================================================================
# Entry point
# ==============================================================================
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="MTCNN + CLAHE offline face preprocessing"
    )
    parser.add_argument("--raw",      default=RAW_DATASET,
                        help="Raw dataset root directory")
    parser.add_argument("--out",      default=OUT_DATASET,
                        help="Output directory for processed dataset")
    parser.add_argument("--no-clahe", action="store_true",
                        help="Disable CLAHE contrast normalisation")
    parser.add_argument("--device",   default="cpu",
                        help="Device for MTCNN: 'cpu' or 'cuda'")

    # parse_known_args() ignores Jupyter/Kaggle kernel args in sys.argv
    args, _ = parser.parse_known_args()

    mtcnn    = load_mtcnn(device=args.device)
    out_path = process_dataset(
        raw_root  = args.raw,
        out_root  = args.out,
        mtcnn     = mtcnn,
        use_clahe = not args.no_clahe,
    )
    per_class_report(args.raw, out_path)

    print(f"\n[*] Done! Update DATA_DIR in your training scripts to:")
    print(f"      {out_path}\n")


[!] facenet-pytorch not installed.
    Run:  pip install facenet-pytorch
    Falling back to centre-crop only (no face detection).

  Source  : /kaggle/input/datasets/mrsohel/autism-dataset/dataset
  Output  : /kaggle/working/dataset_mtcnn
  CLAHE   : ON
  Padding : 20%  |  Fallback crop: 85%

  [train]  Processed=1386 | Face detected=   0 (  0.0%) | Fallback=1386 | Errors= 0 | Skipped=   0
  [valid]  Processed= 298 | Face detected=   0 (  0.0%) | Fallback=298 | Errors= 0 | Skipped=   0
  [test ]  Processed= 304 | Face detected=   0 (  0.0%) | Fallback=304 | Errors= 0 | Skipped=   0

  [TOTAL]  Processed=1988 | Face detected=0 (0.0%) | Fallback=1988 | Errors=0

────────────────────────────────────────────────────
  Per-class images saved (train split)
────────────────────────────────────────────────────
  anger      [██████████████████████████████]  146/146  (100%)
  fear       [██████████████████████████████]   60/ 60  (100%)
  joy        [██████████████████████████████]  599/599  (10

In [ ]:
"""
=============================================================
  Autism Facial Expression Recognition — Full Kaggle Pipeline
  Train 8 curated models on free Kaggle GPU (T4/P100)
=============================================================
"""


import os, sys, json, time, copy
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler
from torch.amp import autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
import pandas as pd
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: No GPU detected — training will be very slow")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True



# ---- Paths (Kaggle default) ----
# Auto-detect: prefer MTCNN-preprocessed dataset if available
_MTCNN_DIR = "/kaggle/working/dataset_mtcnn"
_RAW_DIR   = "/kaggle/input/datasets/mrsohel/autism-dataset/dataset"
DATA_DIR   = _MTCNN_DIR if os.path.exists(_MTCNN_DIR) else _RAW_DIR
print(f"[*] DATA_DIR = {DATA_DIR}")
OUTPUT_DIR = "/kaggle/working/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Hyperparameters ----
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 80
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 15
MIXUP_ALPHA = 0.4
EMA_DECAY = 0.999
NUM_WORKERS = 2

CLASS_NAMES = ["anger", "fear", "joy", "natural", "sadness", "surprise"]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

# ---- Models to train ----
# Curated 8-model set — one representative per architectural family.
# Selected based on Run 1 results (see new new log.log).
# Dropped: VGG19, ResNet18, SEResNet50, MobileNetV3, GhostNet, MobileViT,
#          CrossViT, ViT-Tiny, EfficientNetV2-M, ConvNeXt-Small (collapsed),
#          CoaT-Lite-Small (collapsed), EfficientNetV2-S.
EXPERIMENTS = [
    # Classic CNNs
    {"model": "vgg16",                        "loss": "focal",     "lr": 1e-3},  # Run 1 best: F1=0.5477
    {"model": "inception_v3",                 "loss": "focal",     "lr": 1e-3},  # Run 1: F1=0.5232
    {"model": "densenet121",                  "loss": "focal",     "lr": 1e-3},  # Run 1: F1=0.5229  (7M params)
    {"model": "mobilenetv2_100",              "loss": "focal",     "lr": 1e-3},  # Run 1: F1=0.4984  (lightweight)
    {"model": "resnet50",                     "loss": "focal",     "lr": 1e-3},  # Run 1: F1=0.4636  (universal baseline)

    # Vision Transformers & Hybrids (need lower LR to prevent collapse)
    {"model": "deit_small_patch16_224",       "loss": "ce_smooth", "lr": 1e-4},  # Run 1: F1=0.5437  (Care-FER Stream B)
    {"model": "vit_base_patch16_224",         "loss": "ce_smooth", "lr": 1e-4},  # Run 1: F1=0.5352
    {"model": "swin_base_patch4_window7_224", "loss": "ce_smooth", "lr": 1e-4},  # Run 1: F1=0.4937  (hierarchical ViT)
]



class FacialExpressionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        self.labels = []
        for class_name in CLASS_NAMES:
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                continue
            class_idx = CLASS_TO_IDX[class_name]
            for img_path in class_dir.iterdir():
                if img_path.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".tiff"):
                    self.samples.append(img_path)
                    self.labels.append(class_idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image = Image.open(self.samples[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


def get_train_transforms(img_size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomGrayscale(p=0.05),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])


def get_val_transforms(img_size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def compute_class_weights(dataset):
    counts = Counter(dataset.labels)
    total = len(dataset.labels)
    return torch.FloatTensor([total / (NUM_CLASSES * counts.get(i, 1)) for i in range(NUM_CLASSES)])


def get_dataloaders(data_dir, batch_size=BATCH_SIZE, img_size=IMG_SIZE):
    train_ds = FacialExpressionDataset(os.path.join(data_dir, "train"), get_train_transforms(img_size))
    val_ds = FacialExpressionDataset(os.path.join(data_dir, "valid"), get_val_transforms(img_size))
    test_ds = FacialExpressionDataset(os.path.join(data_dir, "test"), get_val_transforms(img_size))

    # Weighted sampler for class imbalance
    counts = Counter(train_ds.labels)
    sample_weights = [1.0 / counts[label] for label in train_ds.labels]
    sampler = WeightedRandomSampler(sample_weights, len(train_ds), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    class_weights = compute_class_weights(train_ds)
    return train_loader, val_loader, test_loader, class_weights, train_ds


print("Loading datasets...")
train_loader, val_loader, test_loader, class_weights, train_ds = get_dataloaders(DATA_DIR)
class_weights = class_weights.to(DEVICE)
print(f"Train: {len(train_ds)} images")
print(f"Class weights: {dict(zip(CLASS_NAMES, [f'{w:.2f}' for w in class_weights.cpu()]))}")



class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()


def get_loss_fn(loss_type, class_weights=None):
    if loss_type == "focal":
        return FocalLoss(alpha=class_weights, gamma=2.0)
    elif loss_type == "ce_smooth":
        return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    else:
        return nn.CrossEntropyLoss(weight=class_weights)



MODEL_CONFIGS = {
    "vgg16": {"timm": "vgg16_bn", "size": 224},
    "inception_v3": {"timm": "inception_v3", "size": 299},
    "densenet121": {"timm": "densenet121", "size": 224},
    "mobilenetv2_100": {"timm": "mobilenetv2_100", "size": 224},
    "resnet50": {"timm": "resnet50", "size": 224},
    "deit_small_patch16_224": {"timm": "deit_small_patch16_224", "size": 224},
    "vit_base_patch16_224": {"timm": "vit_base_patch16_224.augreg_in21k", "size": 224},
    "swin_base_patch4_window7_224": {"timm": "swin_base_patch4_window7_224", "size": 224},
}


def get_model(name, pretrained=True):
    cfg = MODEL_CONFIGS[name]
    try:
        try:
            model = timm.create_model(cfg["timm"], pretrained=pretrained, num_classes=NUM_CLASSES,
                                      drop_rate=0.3, drop_path_rate=0.2)
        except TypeError:
            model = timm.create_model(cfg["timm"], pretrained=pretrained, num_classes=NUM_CLASSES)
    except RuntimeError as e:
        if pretrained and "pretrained" in str(e).lower():
            print(f"Warning: No pretrained weights for {name}. Using random init.")
            return get_model(name, pretrained=False)
        raise
    return model, cfg["size"]



class EMA:
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = (1 - self.decay) * p.data + self.decay * self.shadow[n]

    def apply_shadow(self):
        self.backup = {n: p.data.clone() for n, p in self.model.named_parameters() if p.requires_grad}
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                p.data = self.shadow[n]

    def restore(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                p.data = self.backup[n]


def mixup_data(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam



def compute_metrics(y_true, y_pred):
    labels_list = list(range(NUM_CLASSES))
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0, labels=labels_list),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0, labels=labels_list),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0, labels=labels_list),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels_list).tolist(),
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0, labels=labels_list),
        "per_class_f1": dict(zip(CLASS_NAMES, [float(f) for f in f1_score(y_true, y_pred, average=None, zero_division=0, labels=labels_list)])),
    }



def train_one_epoch(model, loader, criterion, optimizer, scaler, ema, use_mixup):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=DEVICE.type, dtype=torch.float16 if DEVICE.type == "cuda" else torch.bfloat16):
            if use_mixup:
                mixed, ya, yb, lam = mixup_data(images, labels)
                outputs = model(mixed)
                loss = lam * criterion(outputs, ya) + (1 - lam) * criterion(outputs, yb)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        if ema:
            ema.update()

        if not use_mixup:
            correct += outputs.argmax(1).eq(labels).sum().item()
        else:
            correct += (lam * outputs.argmax(1).eq(ya).sum().item() + (1 - lam) * outputs.argmax(1).eq(yb).sum().item())
        total += labels.size(0)
        total_loss += loss.item() * images.size(0)

    return total_loss / total, correct / total if total else 0


@torch.no_grad()
def evaluate(model, loader, criterion, ema=None):
    if ema:
        ema.apply_shadow()
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        total_loss += criterion(outputs, labels).item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    if ema:
        ema.restore()
    return total_loss / len(all_preds), [int(p) for p in all_preds], [int(l) for l in all_labels], np.array(all_probs)



def plot_confusion_matrix(y_true, y_pred, path, name):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, vmin=0, vmax=1)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{name} — Confusion Matrix")
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


def plot_curves(history, path, name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history["epochs"], history["train_loss"], "b-", label="Train", lw=2)
    axes[0].plot(history["epochs"], history["val_loss"], "r-", label="Val", lw=2)
    axes[0].set(xlabel="Epoch", ylabel="Loss", title=f"{name} — Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history["epochs"], history["train_acc"], "b-", label="Train", lw=2)
    axes[1].plot(history["epochs"], history["val_acc"], "r-", label="Val", lw=2)
    axes[1].set(xlabel="Epoch", ylabel="Accuracy", title=f"{name} — Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


def plot_f1_bars(per_class_f1, path, name):
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(per_class_f1.keys(), per_class_f1.values(), color=sns.color_palette("viridis", len(per_class_f1)),
                  edgecolor="black", lw=0.5)
    ax.set_ylabel("F1-Score"); ax.set_title(f"{name} — Per-Class F1"); ax.set_ylim(0, 1.0)
    for b, v in zip(bars, per_class_f1.values()):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)



all_results = {}
global_test_labels = None

for exp in EXPERIMENTS:
    name = exp["model"]
    loss_type = exp["loss"]

    print(f"\n{'='*60}")
    print(f"  TRAINING: {name} | Loss: {loss_type}")
    print(f"{'='*60}")

    model, input_size = get_model(name, pretrained=True)
    model = model.to(DEVICE)

    # Rebuild dataloaders (transforms are recreated each time for safety)
    train_loader, val_loader, test_loader, class_weights, _ = get_dataloaders(DATA_DIR, img_size=input_size)
    class_weights = class_weights.to(DEVICE)
    total_p = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {total_p:,}")

    criterion = get_loss_fn(loss_type, class_weights)

    # Differential learning rates
    backbone, head = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in n for k in ("classifier", "head", "fc")):
            head.append(p)
        else:
            backbone.append(p)

    exp_lr = exp.get("lr", LEARNING_RATE)
    optimizer = torch.optim.AdamW([
        {"params": backbone, "lr": exp_lr * 0.1},
        {"params": head, "lr": exp_lr},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = GradScaler(enabled=(DEVICE.type == "cuda"))
    ema = EMA(model, EMA_DECAY)

    history = {"epochs": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_f1 = 0.0
    patience_counter = 0

    t0 = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, ema, True)
        scheduler.step()
        val_loss, val_preds, val_labels, _ = evaluate(model, val_loader, criterion, ema)
        val_m = compute_metrics(val_labels, val_preds)

        history["epochs"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_m["accuracy"])

        elapsed = time.time() - t0
        print(
            f"  Epoch {epoch:3d}/{NUM_EPOCHS} | "
            f"TrL {train_loss:.4f} TrA {train_acc:.4f} | "
            f"VlL {val_loss:.4f} VaA {val_m['accuracy']:.4f} F1 {val_m['f1_macro']:.4f} | "
            f"{elapsed/60:.1f}min"
        )

        if val_m["f1_macro"] > best_f1:
            best_f1 = val_m["f1_macro"]
            patience_counter = 0
            torch.save({"epoch": epoch, "state_dict": model.state_dict(),
                        "ema": ema.shadow, "args": exp}, f"{OUTPUT_DIR}/{name}_best.pth")
            print(f"    >> New best F1: {best_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"    >> Early stopping at epoch {epoch}")
                break

    # --- Test evaluation ---
    ckpt = torch.load(f"{OUTPUT_DIR}/{name}_best.pth", map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["state_dict"])
    ema.shadow = ckpt["ema"]

    test_loss, test_preds, test_labels, test_probs = evaluate(model, test_loader, criterion, ema)
    test_m = compute_metrics(test_labels, test_preds)

    print(f"\n  TEST — Acc: {test_m['accuracy']:.4f} | F1: {test_m['f1_macro']:.4f} | Prec: {test_m['precision_macro']:.4f} | Rec: {test_m['recall_macro']:.4f}")
    print(test_m["report"])

    # Save everything
    model_dir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(model_dir, exist_ok=True)
    with open(f"{model_dir}/test_metrics.json", "w") as f:
        json.dump(test_m, f, indent=2)
    with open(f"{model_dir}/history.json", "w") as f:
        json.dump(history, f, indent=2)
    plot_confusion_matrix(test_labels, test_preds, f"{model_dir}/confusion_matrix.png", name)
    plot_curves(history, f"{model_dir}/training_curves.png", name)
    plot_f1_bars(test_m["per_class_f1"], f"{model_dir}/f1_per_class.png", name)

    all_results[name] = test_m
    all_results[name]["params"] = total_p
    all_results[name]["time_min"] = (time.time() - t0) / 60
    all_results[name]["test_probs"] = test_probs
    all_results[name]["test_preds"] = test_preds
    global_test_labels = test_labels

    print(f"  Saved to {model_dir}/")
    torch.cuda.empty_cache()

    # --- Explicit Cleanup ---
    # Prevents "Too many open files" and DataLoader worker process leaks across experiments on Kaggle
    del model, optimizer, scheduler, scaler, criterion, ema
    del train_loader, val_loader, test_loader
    import gc
    gc.collect()



print(f"\n{'='*70}")
print("  FINAL MODEL COMPARISON")
print(f"{'='*70}")

models_sorted = sorted(all_results.keys(), key=lambda k: all_results[k]["f1_macro"], reverse=True)
top_5_models = models_sorted[:min(5, len(models_sorted))]

header = f"{'Model':<35} {'Acc':>8} {'F1':>8} {'Prec':>8} {'Rec':>8} {'Params':>12}"
print(header)
print("-" * len(header))
for name in models_sorted:
    r = all_results[name]
    print(f"{name:<35} {r['accuracy']:>8.4f} {r['f1_macro']:>8.4f} {r['precision_macro']:>8.4f} {r['recall_macro']:>8.4f} {r.get('params',0):>12,}")

# Remove heavy arrays before saving JSON
json_results = copy.deepcopy(all_results)
for k in json_results:
    json_results[k].pop("test_probs", None)
    json_results[k].pop("test_preds", None)
with open(f"{OUTPUT_DIR}/comparison.json", "w") as f:
    json.dump(json_results, f, indent=2)

COMPARISON_DIR = os.path.join(OUTPUT_DIR, "paper_figures")
os.makedirs(COMPARISON_DIR, exist_ok=True)

# 1. Grouped Bar Chart (Acc, F1, Prec, Rec) for Top Models
df_metrics = []
for m in top_5_models:
    df_metrics.extend([
        {"Model": m, "Metric": "Accuracy", "Score": all_results[m]["accuracy"]},
        {"Model": m, "Metric": "F1-Macro", "Score": all_results[m]["f1_macro"]},
        {"Model": m, "Metric": "Precision", "Score": all_results[m]["precision_macro"]},
        {"Model": m, "Metric": "Recall", "Score": all_results[m]["recall_macro"]},
    ])
df_metrics = pd.DataFrame(df_metrics)
fig1 = plt.figure(figsize=(12, 6))
sns.barplot(data=df_metrics, x="Model", y="Score", hue="Metric", palette="Set2")
plt.ylim(0, 1.0)
plt.title("Top 5 Models - Performance Metrics")
plt.tight_layout()
plt.savefig(f"{COMPARISON_DIR}/1_grouped_bar_metrics.png", dpi=300)
plt.close(fig1)

# 2. Box Plot of Per-Class F1 Scores
f1_data = []
for m in models_sorted:
    for cls_name, f1 in all_results[m]["per_class_f1"].items():
        f1_data.append({"Model": m, "Class": cls_name, "F1": f1})
df_f1 = pd.DataFrame(f1_data)
fig2 = plt.figure(figsize=(14, 8))
sns.boxplot(data=df_f1, x="F1", y="Model", palette="coolwarm")
plt.title("Distribution of Per-Class F1 Scores across Models")
plt.xlabel("F1-Score")
plt.tight_layout()
plt.savefig(f"{COMPARISON_DIR}/2_boxplot_f1_distributions.png", dpi=300)
plt.close(fig2)

# 3. Macro ROC Curve (Top 5 Models)
fig3 = plt.figure(figsize=(10, 8))
Y_test_bin = label_binarize(global_test_labels, classes=list(range(NUM_CLASSES)))
for m in top_5_models:
    probs = all_results[m]["test_probs"]
    fpr, tpr, _ = roc_curve(Y_test_bin.ravel(), probs.ravel())
    macro_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{m} (AUC = {macro_auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Macro-Average ROC Curve (Top 5 Models)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig(f"{COMPARISON_DIR}/3_roc_curve.png", dpi=300)
plt.close(fig3)

# 4. Precision-Recall Curve (Top 5 Models)
fig4 = plt.figure(figsize=(10, 8))
for m in top_5_models:
    probs = all_results[m]["test_probs"]
    prec, rec, _ = precision_recall_curve(Y_test_bin.ravel(), probs.ravel())
    ap = average_precision_score(Y_test_bin, probs, average="macro")
    plt.plot(rec, prec, lw=2, label=f"{m} (AP = {ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Macro-Average Precision-Recall Curve (Top 5 Models)")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.savefig(f"{COMPARISON_DIR}/4_pr_curve.png", dpi=300)
plt.close(fig4)

# 5. Radar Chart (Spider Chart)
categories = ['Accuracy', 'F1-Macro', 'Precision', 'Recall']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
fig5, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], categories)
ax.set_rlabel_position(0)
plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], ["0.2", "0.4", "0.6", "0.8", "1.0"], color="grey", size=8)
plt.ylim(0, 1)

for m in top_5_models:
    values = [
        all_results[m]["accuracy"],
        all_results[m]["f1_macro"],
        all_results[m]["precision_macro"],
        all_results[m]["recall_macro"]
    ]
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=m)
    ax.fill(angles, values, alpha=0.1)

plt.title("Radar Chart - Top 5 Models", y=1.1)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.savefig(f"{COMPARISON_DIR}/5_radar_chart.png", dpi=300, bbox_inches="tight")
plt.close(fig5)

# 6. Model Prediction Correlation Heatmap (Ensemble Diversity)
preds_dict = {m: all_results[m]["test_preds"] for m in models_sorted}
df_preds = pd.DataFrame(preds_dict)
corr = df_preds.corr(method="spearman").fillna(0)
fig6 = plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap="coolwarm", vmin=0, vmax=1)
plt.title("Model Prediction Correlation (Spearman)")
plt.tight_layout()
plt.savefig(f"{COMPARISON_DIR}/6_model_correlation_heatmap.png", dpi=300)
plt.close(fig6)

print(f"\nAll results and paper-ready figures saved to {COMPARISON_DIR}/")
print("Download the output dataset from the Kaggle 'Output' tab.")


In [ ]:
"""
========================================================================================
Care-FER: Clinically-Aware Recalibrated Ensemble for Autism Facial Expression Recognition
========================================================================================
Self-contained Kaggle training and evaluation script for the proposed architecture:
1. Dual-Stream Backbone: VGG16 (Local Texture Expert) + DeiT-Small (Global Geometry Expert)
2. Feature Recalibration: Dual Squeeze-and-Excitation (SE) Channel Attention Blocks (r=16)
3. Training Stabilization: Exponential Moving Average (EMA, decay=0.999) + Focal Loss
4. Clinical Inference: 5-View Test-Time Augmentation (TTA, K=5)
5. Clinical Safety: 70% Confidence Uncertainty Rejection Guardrail
6. Publication Figures: Confusion Matrix, Training Curves, Per-Class F1, and Grad-CAM Heatmaps
========================================================================================
To run on Kaggle:
1. Create a Kaggle Notebook with GPU accelerated (Dual T4 or P100).
2. Upload your deduplicated dataset to Kaggle as a dataset named 'autism-facial-expression-recognition'.
3. Copy-paste this entire script into a code cell and run!
========================================================================================
"""

import os
import sys
import copy
import time
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler
from torch.amp import autocast
from torch.nn.utils import clip_grad_norm_
import torchvision.transforms as transforms
import timm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. HYPERPARAMETERS & CONFIGURATION
# ==============================================================================
SEED = 42
NUM_EPOCHS = 160  # Increased: V4 showed val F1 still rising at epoch 78 — dual-stream needs more time
BATCH_SIZE = 16
LEARNING_RATE = 1e-4  # Optimal for Hybrid Transformer-CNN architectures
WEIGHT_DECAY = 1e-4
EMA_DECAY = 0.999
TTA_VIEWS = 5         # K=5 for Test-Time Augmentation
UNCERTAINTY_THRESH = 0.30  # Clinical rejection guardrail (30% confidence for 6-class softmax — 50% caused 100% rejection)
IMG_SIZE = 224
PATIENCE = 20         # Increased: large dual-stream model converges more slowly
MIXUP_ALPHA = 0.4     # MixUp regularization strength
NUM_WORKERS = 2

# Kaggle dataset paths (fallback to local if running locally for testing)
# Dataset path — priority order:
#   1. MTCNN-preprocessed output from Cell 1 (same notebook session)
#   2. Raw Kaggle input dataset
#   3. Local path (for testing)
KAGGLE_MTCNN_DIR  = "/kaggle/working/dataset_mtcnn"
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/mrsohel/autism-dataset/dataset"
LOCAL_DATASET_DIR = r"C:\Users\mrsoh\Documents\Autism-Facial-Expression-Recognition\dataset"

if os.path.exists(KAGGLE_MTCNN_DIR):
    DATASET_DIR = KAGGLE_MTCNN_DIR
elif os.path.exists(KAGGLE_DATASET_DIR):
    DATASET_DIR = KAGGLE_DATASET_DIR
else:
    DATASET_DIR = LOCAL_DATASET_DIR

OUTPUT_DIR = "/kaggle/working/results/care_fer_proposed" if os.path.exists("/kaggle") else "./results/care_fer_proposed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSES = ["anger", "fear", "joy", "natural", "sadness", "surprise"]
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {cls_name: i for i, cls_name in enumerate(CLASSES)}


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Running on Device: {device} | Output Directory: {OUTPUT_DIR}")
print(f"[*] Dataset Directory: {DATASET_DIR}")


# ==============================================================================
# 2. DATASET & CLINICAL AUGMENTATIONS
# ==============================================================================
class AutismFERDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.root_dir = Path(root_dir) / split
        self.transform = transform
        self.samples = []
        self.targets = []
        
        if not self.root_dir.exists():
            raise FileNotFoundError(f"Directory not found: {self.root_dir}")
            
        for cls_idx, cls_name in enumerate(CLASSES):
            cls_dir = self.root_dir / cls_name
            if not cls_dir.exists(): continue
            for img_path in cls_dir.iterdir():
                if img_path.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"):
                    self.samples.append(str(img_path))
                    self.targets.append(cls_idx)
                    
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        img_path = self.samples[idx]
        target = self.targets[idx]
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))
            
        if self.transform:
            img = self.transform(img)
        return img, target

# Standard training transforms (matches baseline run_all_models.py augmentation strength)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

# Standard validation transform
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 5-View Test-Time Augmentation (TTA) transforms generator
def get_tta_transforms():
    base_norm = [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
    return [
        transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE))] + base_norm), # 1. Center original
        transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(p=1.0)] + base_norm), # 2. Flip H
        transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomRotation((5, 5))] + base_norm), # 3. Rotate +5 deg
        transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomRotation((-5, -5))] + base_norm), # 4. Rotate -5 deg
        transforms.Compose([transforms.Resize((int(IMG_SIZE*1.08), int(IMG_SIZE*1.08))), transforms.CenterCrop(IMG_SIZE)] + base_norm), # 5. Slight zoom
    ]

# Offline Face Alignment Preprocessing (Path B)
def align_and_crop_faces(src_dir, dst_dir):
    """
    Standardizes input images by detecting facial bounding boxes with OpenCV Haar Cascade
    and cropping with 15% padding. Removes background noise, neck/shoulders, and scale
    inconsistencies (matching the methodology of top-performing research papers).
    Fallback: Center crop (85%) if face detection fails on extreme lighting/poses.
    """
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    if dst_dir.exists() and sum(1 for f in dst_dir.rglob("*.*") if f.is_file()) > 1000:
        print(f"[*] Aligned dataset already exists at {dst_dir}. Skipping preprocessing.")
        return str(dst_dir)
        
    print(f"[*] Running Face Alignment Preprocessing (OpenCV Haar Cascade + 15% padding)...")
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    alt_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')
    
    total_imgs = 0
    detected_faces = 0
    
    for split in ["train", "valid", "test"]:
        for cls_name in CLASSES:
            s_dir = src_dir / split / cls_name
            d_dir = dst_dir / split / cls_name
            os.makedirs(d_dir, exist_ok=True)
            if not s_dir.exists(): continue
            
            for img_path in s_dir.iterdir():
                if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png", ".bmp"): continue
                total_imgs += 1
                try:
                    img_cv = cv2.imread(str(img_path))
                    if img_cv is None:
                        Image.open(img_path).convert("RGB").save(d_dir / img_path.name)
                        continue
                    
                    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
                    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(30, 30))
                    if len(faces) == 0:
                        faces = alt_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(30, 30))
                        
                    if len(faces) > 0:
                        largest_face = max(faces, key=lambda b: b[2] * b[3])
                        x, y, w, h = largest_face
                        img_h, img_w, _ = img_cv.shape
                        
                        pad_w = int(w * 0.15)
                        pad_h = int(h * 0.15)
                        x1 = max(0, x - pad_w)
                        y1 = max(0, y - pad_h)
                        x2 = min(img_w, x + w + pad_w)
                        y2 = min(img_h, y + h + pad_h)
                        
                        cropped_cv = img_cv[y1:y2, x1:x2]
                        detected_faces += 1
                    else:
                        img_h, img_w, _ = img_cv.shape
                        cw, ch = int(img_w * 0.85), int(img_h * 0.85)
                        x1 = (img_w - cw) // 2
                        y1 = (img_h - ch) // 2
                        cropped_cv = img_cv[y1:y1+ch, x1:x1+cw]
                        
                    cropped_rgb = cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2RGB)
                    Image.fromarray(cropped_rgb).save(d_dir / img_path.name)
                except Exception as e:
                    Image.open(img_path).convert("RGB").save(d_dir / img_path.name)
                    
    print(f"[*] Face Alignment Complete! Aligned {detected_faces}/{total_imgs} images ({detected_faces/max(1, total_imgs)*100:.1f}% success).")
    return str(dst_dir)

# Face alignment disabled: Haar Cascade only succeeded on 64.4% of images;
# the 35.6% fallback center-crops introduced noise and hurt performance.
# ALIGNED_DATASET_DIR = "/kaggle/working/aligned_dataset" if os.path.exists("/kaggle") else "./aligned_dataset"
# DATASET_DIR = align_and_crop_faces(DATASET_DIR, ALIGNED_DATASET_DIR)

# Load Datasets
train_dataset = AutismFERDataset(DATASET_DIR, split="train", transform=train_transform)
val_dataset = AutismFERDataset(DATASET_DIR, split="valid", transform=val_transform)
test_dataset = AutismFERDataset(DATASET_DIR, split="test", transform=val_transform)

# Weighted Random Sampler to overcome 10:1 severe class imbalance on fear
class_counts = Counter(train_dataset.targets)
total_samples = len(train_dataset)
class_weights = {cls_idx: total_samples / (len(class_counts) * count) for cls_idx, count in class_counts.items()}
sample_weights = [class_weights[target] for target in train_dataset.targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=total_samples, replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
train_loader_stage2 = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"[*] Dataset Loaded: Train={len(train_dataset)} | Valid={len(val_dataset)} | Test={len(test_dataset)}")


# ==============================================================================
# 3. PROPOSED ARCHITECTURE: Care-FER (Dual-Stream SE Recalibrated Ensemble)
# ==============================================================================
class SqueezeExcitationBlock(nn.Module):
    """
    Squeeze-and-Excitation (SE) Block (r=16).
    Dynamically recalibrates feature channels to suppress background noise and amplify emotion cues.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (B, C)
        w = self.fc1(x)
        w = self.relu(w)
        w = self.fc2(w)
        w = self.sigmoid(w)
        return x * w


class CareFERModel(nn.Module):
    """
    Care-FER v2 — Redesigned Dual-Stream Architecture:
    - Stream A: VGG16-BN SPATIAL features via forward_features()+GAP
      * Uses conv feature maps (B,512,7,7) → GAP → (B,512)
      * NOT the 4096-d FC pre_logits — spatial features are:
        (a) Smaller (512 vs 4096) = less overfitting on 1386 images
        (b) Proper SE attention on meaningful conv channels
        (c) Enable true Grad-CAM saliency on 7×7 facial regions
    - Stream B: DeiT-Small CLS token (B,384) — global attention geometry
    - Dual SE Blocks: per-stream channel recalibration
    - Head: 896→512→256→6 with GELU + progressive dropout
    """
    def __init__(self, num_classes=6, pretrained=True):
        super().__init__()
        # Stream A: VGG16-BN — spatial conv features (bypasses FC pre_logits)
        vgg = timm.create_model("vgg16_bn", pretrained=pretrained, num_classes=0)
        self.stream_a = vgg
        dim_a = 512  # VGG16-BN last conv block always outputs 512 channels

        # Stream B: DeiT-Small — CLS token (always 384-d)
        deit = timm.create_model("deit_small_patch16_224", pretrained=pretrained, num_classes=0)
        self.stream_b = deit
        dim_b = 384

        print(f"[*] Stream A (VGG16-BN spatial+GAP): {dim_a}-d | Stream B (DeiT-S CLS): {dim_b}-d | Combined: {dim_a+dim_b}-d")

        self.se_a = SqueezeExcitationBlock(dim_a, reduction=16)   # 512→32
        self.se_b = SqueezeExcitationBlock(dim_b, reduction=16)   # 384→24

        # Deeper classification head: 896→512→256→6
        combined_dim = dim_a + dim_b  # 896
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(combined_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(p=0.25),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        # Stream A: spatial conv features via forward_features() then Global Average Pool
        feat_map_a = self.stream_a.forward_features(x)  # (B, 512, 7, 7)
        feat_a = feat_map_a.mean(dim=[2, 3])             # GAP → (B, 512)

        # Stream B: DeiT CLS token
        feat_b = self.stream_b(x)                        # (B, 384)

        # SE recalibration per stream
        rec_a = self.se_a(feat_a)  # (B, 512)
        rec_b = self.se_b(feat_b)  # (B, 384)

        # Concatenate and classify
        fused = torch.cat([rec_a, rec_b], dim=1)  # (B, 896)
        return self.classifier(fused)

model = CareFERModel(num_classes=NUM_CLASSES, pretrained=True).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"[*] Proposed Model Created: Care-FER | Total Parameters: {total_params:,} ({total_params/1e6:.1f}M)")


# ==============================================================================
# 4. LOSS FUNCTION, EMA & TRAINING UTILITIES
# ==============================================================================
class FocalLoss(nn.Module):
    """Focal Loss for severe clinical class imbalance (prioritizing distress emotions).
    alpha: per-class weight tensor (inverse-frequency) or None for uniform weighting.
    """
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha  # Tensor of shape (num_classes,) or None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Pass per-class weights into CE so rare classes (fear, anger) get upweighted
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        return focal_loss.sum()


class ModelEMA:
    """Exponential Moving Average (EMA) for training weight stabilization."""
    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model)
        self.module.eval()
        self.decay = decay
        for p in self.module.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            for ema_param, model_param in zip(self.module.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(model_param.data, alpha=1 - self.decay)

# Compute inverse-frequency class weights (matches run_all_models.py baseline formula)
# This upweights rare classes (fear: weight ~3.89) to boost distress emotion recall
_class_counts_list = [class_counts[i] for i in range(NUM_CLASSES)]
_class_weights = torch.tensor(
    [total_samples / (NUM_CLASSES * c) for c in _class_counts_list], dtype=torch.float32
).to(device)

# V6 Boost: Strongly upweight sadness (x2.0) and fear (x1.2) to overcome recall collapse and imbalance
_sadness_idx = CLASSES.index("sadness")
_fear_idx = CLASSES.index("fear")
_class_weights[_sadness_idx] *= 2.0  # V5 sadness recall was only 15.9% (computed weight 0.72 was too low)
_class_weights[_fear_idx] *= 1.2     # V5 fear recall was 28.6%
print(f"[*] Focal Loss Class Weights (V6 Boost): { {CLASSES[i]: f'{_class_weights[i].item():.2f}' for i in range(NUM_CLASSES)} }")
loss_fn = FocalLoss(alpha=_class_weights, gamma=1.5)  # gamma 1.5: softer than 2.0, avoids over-suppressing easy samples

# Differential learning rate: backbones get 0.1x LR, SE blocks & head get full LR
backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if "classifier" in name or "se_a" in name or "se_b" in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LEARNING_RATE * 0.1},  # 1e-5 for pretrained backbones (matches baseline differential LR ratio)
    {"params": head_params, "lr": LEARNING_RATE},
], weight_decay=WEIGHT_DECAY)

# Warmup (10 epochs linear) + Cosine Decay to 0.
# CosineAnnealingWarmRestarts caused LR spikes every T_0=10 epochs which destabilizes DeiT.
_WARMUP_EPOCHS = 10
def _lr_lambda(epoch):
    if epoch < _WARMUP_EPOCHS:
        return float(epoch + 1) / float(_WARMUP_EPOCHS)   # 0.1 → 1.0 linearly
    progress = float(epoch - _WARMUP_EPOCHS) / float(max(1, NUM_EPOCHS - _WARMUP_EPOCHS))
    return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))  # cosine to 1% of LR
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=_lr_lambda)
ema_model = ModelEMA(model, decay=EMA_DECAY)
scaler = GradScaler()


# MixUp augmentation (matches baseline pipeline)
def mixup_data(x, y, alpha=MIXUP_ALPHA):
    """MixUp: blends pairs of training images and labels for regularization."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


# ==============================================================================
# 5. TRAINING & VALIDATION LOOP
# ==============================================================================
print("\n" + "="*70)
print("  STARTING CLINICAL TRAINING: Care-FER Architecture")
print("="*70)

best_val_f1 = 0.0
best_model_path = os.path.join(OUTPUT_DIR, "care_fer_best.pth")
train_losses, val_losses, train_accs, val_accs, val_f1s = [], [], [], [], []

patience_counter = 0
start_time = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    # --- TRAINING (with MixUp) ---
    model.train()
    running_loss, running_correct, total_train = 0.0, 0, 0
    
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
            mixed_imgs, ya, yb, lam = mixup_data(imgs, targets)
            outputs = model(mixed_imgs)
            loss = lam * loss_fn(outputs, ya) + (1 - lam) * loss_fn(outputs, yb)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        ema_model.update(model)
        
        running_loss += loss.item() * imgs.size(0)
        # MixUp-aware accuracy: weighted sum of matches against both mixed labels
        running_correct += (lam * outputs.argmax(1).eq(ya).sum().item() +
                           (1 - lam) * outputs.argmax(1).eq(yb).sum().item())
        total_train += imgs.size(0)
        
    scheduler.step()
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = running_correct / total_train
    
    # --- VALIDATION (using EMA weights) ---
    ema_model.module.eval()
    val_loss, val_correct, total_val = 0.0, 0, 0
    val_preds_list, val_targets_list = [], []
    
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            with autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
                outputs = ema_model.module(imgs)
                loss = loss_fn(outputs, targets)
                
            val_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == targets).sum().item()
            total_val += imgs.size(0)
            
            val_preds_list.extend(preds.cpu().numpy())
            val_targets_list.extend(targets.cpu().numpy())
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = val_correct / total_val
    epoch_val_f1 = f1_score(val_targets_list, val_preds_list, average="macro")
    
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accs.append(epoch_train_acc)
    val_accs.append(epoch_val_acc)
    val_f1s.append(epoch_val_f1)
    
    # Log progress every epoch (matches baseline verbosity)
    elapsed = (time.time() - start_time) / 60
    print(f"  Epoch {epoch:3d}/{NUM_EPOCHS} | TrL {epoch_train_loss:.4f} TrA {epoch_train_acc:.4f} | "
          f"VlL {epoch_val_loss:.4f} VaA {epoch_val_acc:.4f} F1 {epoch_val_f1:.4f} | {elapsed:.1f}min")
        
    if epoch_val_f1 > best_val_f1:
        best_val_f1 = epoch_val_f1
        patience_counter = 0
        print(f"    >> New best F1: {best_val_f1:.4f}")
        torch.save({
            "epoch": epoch,
            "model_state_dict": ema_model.module.state_dict(),
            "val_f1": best_val_f1,
            "val_acc": epoch_val_acc,
        }, best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch} (no F1 improvement for {PATIENCE} epochs)")
            break

total_time = (time.time() - start_time) / 60
print(f"\n[*] Stage 1 Complete in {total_time:.1f} minutes! Best Validation Macro F1: {best_val_f1:.4f}")
print(f"[*] Stage 1 checkpoint saved to: {best_model_path}")

# ==============================================================================
# 5.5 STAGE 2: CLASSIFIER FINE-TUNING (DECOUPLED TRAINING)
# ==============================================================================
print("\n" + "="*70)
print("  STARTING STAGE 2: Classifier Fine-Tuning (Frozen Backbone)")
print("="*70)

# Load best model from Stage 1
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

# Freeze backbone, unfreeze head
for name, param in model.named_parameters():
    if any(k in name for k in ("classifier", "se_a", "se_b")):
        param.requires_grad = True
    else:
        param.requires_grad = False

head_params = [p for p in model.parameters() if p.requires_grad]
optimizer_stage2 = torch.optim.AdamW(head_params, lr=1e-4, weight_decay=WEIGHT_DECAY)
ema_model_stage2 = ModelEMA(model, decay=EMA_DECAY)

STAGE2_EPOCHS = 20
patience_counter_s2 = 0
start_time_s2 = time.time()
best_val_f1_s2 = best_val_f1

for epoch in range(1, STAGE2_EPOCHS + 1):
    model.train()
    running_loss, running_correct, total_train = 0.0, 0, 0
    
    # USE THE BALANCED SAMPLER FOR STAGE 2
    for imgs, targets in train_loader_stage2:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer_stage2.zero_grad(set_to_none=True)
        
        with autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
            mixed_imgs, ya, yb, lam = mixup_data(imgs, targets)
            outputs = model(mixed_imgs)
            loss = lam * loss_fn(outputs, ya) + (1 - lam) * loss_fn(outputs, yb)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer_stage2)
        clip_grad_norm_(head_params, max_norm=5.0)
        scaler.step(optimizer_stage2)
        scaler.update()
        ema_model_stage2.update(model)
        
        running_loss += loss.item() * imgs.size(0)
        running_correct += (lam * outputs.argmax(1).eq(ya).sum().item() +
                           (1 - lam) * outputs.argmax(1).eq(yb).sum().item())
        total_train += imgs.size(0)
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = running_correct / total_train
    
    ema_model_stage2.module.eval()
    val_loss, val_correct, total_val = 0.0, 0, 0
    val_preds_list, val_targets_list = [], []
    
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            with autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
                outputs = ema_model_stage2.module(imgs)
                loss = loss_fn(outputs, targets)
                
            val_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == targets).sum().item()
            total_val += imgs.size(0)
            
            val_preds_list.extend(preds.cpu().numpy())
            val_targets_list.extend(targets.cpu().numpy())
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = val_correct / total_val
    epoch_val_f1 = f1_score(val_targets_list, val_preds_list, average="macro")
    
    elapsed = (time.time() - start_time_s2) / 60
    print(f"  Stage 2 Epoch {epoch:3d}/{STAGE2_EPOCHS} | TrL {epoch_train_loss:.4f} TrA {epoch_train_acc:.4f} | "
          f"VlL {epoch_val_loss:.4f} VaA {epoch_val_acc:.4f} F1 {epoch_val_f1:.4f} | {elapsed:.1f}min")
        
    if epoch_val_f1 > best_val_f1_s2:
        best_val_f1_s2 = epoch_val_f1
        patience_counter_s2 = 0
        print(f"    >> New best F1 (Stage 2): {best_val_f1_s2:.4f}")
        torch.save({
            "epoch": epoch,
            "model_state_dict": ema_model_stage2.module.state_dict(),
            "val_f1": best_val_f1_s2,
            "val_acc": epoch_val_acc,
        }, best_model_path)
    else:
        patience_counter_s2 += 1
        if patience_counter_s2 >= PATIENCE:
            print(f"  Early stopping Stage 2 at epoch {epoch}")
            break

print(f"\n[*] Stage 2 Complete! Final Best Validation Macro F1: {best_val_f1_s2:.4f}")


# ==============================================================================
# 6. CLINICAL EVALUATION WITH 5-VIEW TTA & UNCERTAINTY GUARDRAIL
# ==============================================================================
print("\n" + "="*70)
print(f"  EVALUATING ON TEST SET ({len(test_dataset)} Images) WITH 5-VIEW TTA & GUARDRAILS")
print("="*70)

# Load best checkpoint
checkpoint = torch.load(best_model_path, map_location=device)
eval_model = CareFERModel(num_classes=NUM_CLASSES, pretrained=False).to(device)
eval_model.load_state_dict(checkpoint["model_state_dict"])
eval_model.eval()

tta_transforms = get_tta_transforms()
test_preds, test_targets, test_confidences = [], [], []

with torch.no_grad():
    for img_path, target in zip(test_dataset.samples, test_dataset.targets):
        try:
            raw_img = Image.open(img_path).convert("RGB")
        except Exception:
            raw_img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))
            
        # Run 5 augmented crops/flips through model and average probabilities
        tta_probs = torch.zeros((1, NUM_CLASSES), device=device)
        for t_idx, t_form in enumerate(tta_transforms):
            t_img = t_form(raw_img).unsqueeze(0).to(device)
            with autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
                logits = eval_model(t_img)
                probs = F.softmax(logits, dim=1)
            tta_probs += probs
            
        tta_probs /= len(tta_transforms)
        max_prob, pred_cls = torch.max(tta_probs, dim=1)
        
        test_preds.append(pred_cls.item())
        test_targets.append(target)
        test_confidences.append(max_prob.item())

# Standard Test Metrics
test_acc = accuracy_score(test_targets, test_preds)
test_f1 = f1_score(test_targets, test_preds, average="macro")
test_prec = precision_score(test_targets, test_preds, average="macro", zero_division=0)
test_rec = recall_score(test_targets, test_preds, average="macro", zero_division=0)

print(f"\n[★] OVERALL TEST RESULTS (with 5-View TTA):")
print(f"    Accuracy:      {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"    Macro F1:      {test_f1:.4f}")
print(f"    Precision:     {test_prec:.4f}")
print(f"    Recall:        {test_rec:.4f}\n")

print(classification_report(test_targets, test_preds, target_names=CLASSES, digits=4))

# Distress Emotions Recall Audit (Clinical Safety Check)
distress_classes = ["anger", "fear", "sadness"]
print("-" * 50)
print("CLINICAL SAFETY AUDIT: Distress Emotion Recall (Sensitivity)")
print("-" * 50)
report_dict = classification_report(test_targets, test_preds, target_names=CLASSES, output_dict=True)
for d_cls in distress_classes:
    rec = report_dict[d_cls]["recall"]
    sup = report_dict[d_cls]["support"]
    print(f"  [{d_cls.upper():<8}] Recall: {rec*100:.1f}% (Support: {sup} images)")
print("-" * 50)

# Clinical Uncertainty Rejection Guardrail (>70% Confidence Check)
high_conf_indices = [i for i, conf in enumerate(test_confidences) if conf >= UNCERTAINTY_THRESH]
low_conf_indices = [i for i, conf in enumerate(test_confidences) if conf < UNCERTAINTY_THRESH]
rejection_rate = len(low_conf_indices) / len(test_confidences) * 100

print(f"\n[★] CLINICAL UNCERTAINTY GUARDRAIL (Threshold = {UNCERTAINTY_THRESH*100:.0f}% Confidence):")
print(f"    High-Confidence Diagnostic Diagnoses: {len(high_conf_indices)} / {len(test_confidences)} images")
print(f"    Flagged for Caregiver Review (Low Conf): {len(low_conf_indices)} images ({rejection_rate:.1f}% Rejection Rate)")

if high_conf_indices:
    hc_targets = [test_targets[i] for i in high_conf_indices]
    hc_preds = [test_preds[i] for i in high_conf_indices]
    hc_acc = accuracy_score(hc_targets, hc_preds)
    hc_f1 = f1_score(hc_targets, hc_preds, average="macro")
    print(f"    -> High-Confidence Subset Accuracy: {hc_acc*100:.2f}% | Macro F1: {hc_f1:.4f}")


# ==============================================================================
# 7. PUBLICATION-READY PLOTS & ARTIFACT GENERATION
# ==============================================================================
print("\n[*] Generating publication-ready figures in 300 DPI...")
sns.set_theme(style="whitegrid", font_scale=1.1)

# Plot 1: Training & Validation Curves
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Focal Loss", color="royalblue", lw=2)
plt.plot(val_losses, label="Val Focal Loss (EMA)", color="darkorange", lw=2)
plt.title("Care-FER: Training & Validation Loss", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot([a * 100 for a in train_accs], label="Train Accuracy", color="royalblue", lw=2)
plt.plot([a * 100 for a in val_accs], label="Val Accuracy (EMA)", color="darkorange", lw=2)
plt.title("Care-FER: Accuracy Curves (%)", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "1_training_curves.png"), dpi=300)
plt.close()

# Plot 2: Confusion Matrix (Normalized)
cm = confusion_matrix(test_targets, test_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, cbar=True)
plt.title("Care-FER: Normalized Test Confusion Matrix (TTA K=5)", fontweight="bold", pad=15)
plt.ylabel("True Emotion Label")
plt.xlabel("Predicted Emotion Label")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "2_test_confusion_matrix.png"), dpi=300)
plt.close()

# Plot 3: Per-Class F1, Precision, and Recall Bar Chart
metrics_df = pd.DataFrame({
    "Class": CLASSES,
    "Precision": [report_dict[c]["precision"] for c in CLASSES],
    "Recall": [report_dict[c]["recall"] for c in CLASSES],
    "F1-Score": [report_dict[c]["f1-score"] for c in CLASSES],
}).melt(id_vars="Class", var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_df, x="Class", y="Score", hue="Metric", palette="Set2")
plt.title("Care-FER: Per-Class Diagnostic Performance on Test Set", fontweight="bold")
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.legend(title="Metric")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "3_per_class_metrics.png"), dpi=300)
plt.close()

print(f"[*] All publication charts saved to: {OUTPUT_DIR}")


# ==============================================================================
# 8. GRAD-CAM HEATMAPS (Clinical Explainability — VGG16 Stream A)
# ==============================================================================
print("\n[*] Generating Grad-CAM explainability heatmaps (VGG16 Stream A)...")

class GradCAM:
    """
    Grad-CAM using tensor-level gradient hooks (avoids register_full_backward_hook
    which conflicts with VGG16's inplace ReLU operations in pre_logits FC layers).
    Hooks the last Conv2d of stream_a; uses activation.register_hook() to capture
    gradients without triggering the BackwardHookFunctionBackward inplace error.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self._activation = None
        self._fwd_handle = target_layer.register_forward_hook(self._fwd_hook)

    def _fwd_hook(self, module, inp, output):
        self._activation = output          # live tensor; retain_grad called in generate()

    def generate(self, input_tensor, target_class):
        """Returns a normalized CAM array (H, W) for the given target class."""
        self.model.eval()
        _grad_holder = [None]

        with torch.enable_grad():
            # Fresh input that carries gradients through the graph
            x = input_tensor.detach().clone().requires_grad_(True)
            out = self.model(x)                    # forward — triggers _fwd_hook

            # Retain grad on the (non-leaf) activation so its hook fires
            if self._activation is not None and self._activation.requires_grad:
                self._activation.retain_grad()
                _h = self._activation.register_hook(
                    lambda g: _grad_holder.__setitem__(0, g.detach())
                )
            else:
                return np.zeros((7, 7))            # fallback: no grad path to target layer

            self.model.zero_grad()
            out[0, target_class].backward()
            _h.remove()

        grads = _grad_holder[0]                    # (1, C, H, W) or None
        acts  = self._activation.detach()          # (1, C, H, W)

        if grads is None:
            return np.zeros((acts.shape[2], acts.shape[3]))

        # Global-average-pool gradients → channel importance weights
        weights = grads.mean(dim=[2, 3], keepdim=True)          # (1, C, 1, 1)
        cam = (weights * acts).sum(dim=1).squeeze().cpu().numpy()  # (H, W)
        cam = np.maximum(cam, 0)                   # ReLU
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

    def remove(self):
        self._fwd_handle.remove()


# Find the last Conv2d inside stream_a (VGG16 feature extractor)
_last_conv = None
for _module in eval_model.stream_a.modules():
    if isinstance(_module, nn.Conv2d):
        _last_conv = _module  # iterates in order; ends on the last one

if _last_conv is None:
    print("[!] No Conv2d found in stream_a — Grad-CAM skipped.")
else:
    grad_cam = GradCAM(eval_model, _last_conv)

    # Collect one sample per class from the test set
    _seen, _gradcam_samples = set(), []
    for _path, _cls in zip(test_dataset.samples, test_dataset.targets):
        if _cls not in _seen:
            _seen.add(_cls)
            _gradcam_samples.append((_path, _cls))
        if len(_seen) == NUM_CLASSES:
            break

    fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(3 * NUM_CLASSES, 6))
    fig.suptitle(
        "Care-FER Grad-CAM: VGG16 Stream Discriminative Facial Regions",
        fontsize=13, fontweight="bold",
    )

    for col, (_img_path, _true_cls) in enumerate(_gradcam_samples):
        # --- Original image ---
        _raw = Image.open(_img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        _tensor = val_transform(_raw).unsqueeze(0).to(device).requires_grad_(True)

        # --- Grad-CAM ---
        _cam = grad_cam.generate(_tensor, target_class=_true_cls)

        # Upsample CAM to full image resolution
        _cam_up = np.array(
            Image.fromarray((_cam * 255).astype(np.uint8)).resize(
                (IMG_SIZE, IMG_SIZE), Image.BILINEAR
            )
        ) / 255.0

        # Blend heatmap over original image
        _img_np = np.array(_raw) / 255.0
        _heatmap = plt.cm.jet(_cam_up)[:, :, :3]   # RGBA → RGB via colormap
        _overlay = np.clip(0.5 * _img_np + 0.5 * _heatmap, 0, 1)

        # Top row: original
        axes[0, col].imshow(_raw)
        axes[0, col].set_title(CLASSES[_true_cls].capitalize(), fontsize=10, fontweight="bold")
        axes[0, col].axis("off")

        # Bottom row: Grad-CAM overlay
        axes[1, col].imshow(_overlay)
        axes[1, col].set_title("Grad-CAM", fontsize=9)
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "4_gradcam_heatmaps.png"), dpi=300, bbox_inches="tight")
    plt.close()
    grad_cam.remove()
    print(f"[*] Grad-CAM heatmaps saved to: {OUTPUT_DIR}/4_gradcam_heatmaps.png")

print("========================================================================================")
print("  Care-FER Evaluation Complete! Ready for Paper Publication.")
print("========================================================================================")


[*] Running on Device: cuda | Output Directory: /kaggle/working/results/care_fer_proposed
[*] Dataset Directory: /kaggle/working/dataset_mtcnn
[*] Dataset Loaded: Train=1386 | Valid=298 | Test=304
[*] Stream A (VGG16-BN spatial+GAP): 512-d | Stream B (DeiT-S CLS): 384-d | Combined: 896-d
[*] Proposed Model Created: Care-FER | Total Parameters: 156,579,526 (156.6M)
[*] Focal Loss Class Weights (V6 Boost): {'anger': '1.58', 'fear': '4.62', 'joy': '0.39', 'natural': '1.45', 'sadness': '1.44', 'surprise': '2.26'}

  STARTING CLINICAL TRAINING: Care-FER Architecture
  Epoch   1/160 | TrL 1.8993 TrA 0.2187 | VlL 1.8356 VaA 0.1074 F1 0.0795 | 0.4min
    >> New best F1: 0.0795
  Epoch   2/160 | TrL 1.8684 TrA 0.2221 | VlL 1.8354 VaA 0.1074 F1 0.0794 | 0.8min
  Epoch   3/160 | TrL 1.8523 TrA 0.1889 | VlL 1.8349 VaA 0.1074 F1 0.0789 | 1.1min
  Epoch   4/160 | TrL 1.8278 TrA 0.1935 | VlL 1.8343 VaA 0.1040 F1 0.0743 | 1.5min
  Epoch   5/160 | TrL 1.7734 TrA 0.2096 | VlL 1.8336 VaA 0.1074 F1 0.0755